# PodcastGen-Agent
Generate a complete podcast MP3 from any topic.

## Before you start
1. Enable a **GPU** for this notebook session.
2. Allow outbound network access if your host requires it (`git clone` and `pip install`).
3. Run cells **1 and 2** before any other cell.
4. Start with `DURATION = 2`.
5. If the session disconnects, copy `run_id` from the logs into `RUN_ID` and rerun cell 4.
6. Cells 5 and 6 play/download the latest MP3 after a successful generation.


In [ ]:
# Clone or update the repository, then expose shared notebook helpers.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Rashal10/PodcastGen-Agent.git"

def _work_root() -> Path:
    for candidate in (Path("/kaggle/working"), Path("/content")):
        if candidate.is_dir():
            return candidate
    return Path.cwd()


def _resolve_repo_dir() -> Path:
    env_repo = os.environ.get("PODCAST_GEN_REPO")
    if env_repo:
        candidate = Path(env_repo).resolve()
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate
    for candidate in (
        _work_root() / "PodcastGen-Agent",
        Path.cwd(),
        Path.cwd().parent,
    ):
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate.resolve()
    raise RuntimeError("Repository not found. Run cell 1 first.")


def _prepare_repo(repo_dir: Path | None = None) -> Path:
    root = _resolve_repo_dir() if repo_dir is None else Path(repo_dir).resolve()
    os.chdir(root)
    os.environ["PODCAST_GEN_REPO"] = str(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root


def _find_latest_podcast_mp3() -> Path | None:
    roots: list[Path] = []
    seen: set[str] = set()

    def add(path: Path | str) -> None:
        resolved = Path(path).resolve()
        key = str(resolved)
        if key in seen:
            return
        seen.add(key)
        roots.append(resolved)

    repo = Path(os.environ.get("PODCAST_GEN_REPO", Path.cwd())).resolve()
    add(repo / "outputs")
    add(Path.cwd() / "outputs")
    add(_work_root() / "PodcastGen-Agent" / "outputs")
    for content_root in (Path("/content"), Path("/kaggle/working")):
        if not content_root.exists():
            continue
        for manifest in content_root.rglob("run_manifest.json"):
            add(manifest.parent)

    best: Path | None = None
    best_mtime = -1.0
    for root in roots:
        if not root.exists():
            continue
        mp3_files = [p for p in root.rglob("podcast_*.mp3") if p.stat().st_size > 0]
        if not mp3_files:
            mp3_files = [p for p in root.rglob("*.mp3") if p.stat().st_size > 0]
        for path in mp3_files:
            mtime = path.stat().st_mtime
            if mtime > best_mtime:
                best_mtime = mtime
                best = path
    return best


def _download_or_link_file(path: Path) -> None:
    import importlib

    resolved = path.resolve()

    try:
        files_mod = importlib.import_module("google.colab.files")
        files_mod.download(str(resolved))
        return
    except Exception:
        pass

    work = _work_root()
    dest = work / resolved.name
    if resolved != dest.resolve():
        dest.write_bytes(resolved.read_bytes())
        print(f"Copied to {dest}")

    try:
        from IPython.display import FileLink, display

        try:
            link_target = str(dest.relative_to(Path.cwd()))
        except ValueError:
            link_target = dest.name
        display(FileLink(link_target))
        print("Use the link above, or download the file from the notebook output files.")
    except Exception:
        print(f"File ready at: {dest}")

WORK_ROOT = _work_root()
REPO_DIR = WORK_ROOT / "PodcastGen-Agent"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

REPO_DIR = _prepare_repo(REPO_DIR)
print("Work root:", WORK_ROOT)
print("Repository ready:", REPO_DIR)
print("Package present:", (REPO_DIR / "podcast_gen_agent").is_dir())


In [ ]:
# Install dependencies and verify them in this same kernel (no second Python process).
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(os.environ.get("PODCAST_GEN_REPO", Path.cwd())).resolve()
if not (REPO_DIR / "requirements-colab.txt").is_file():
    raise RuntimeError(f"Repository not ready at {REPO_DIR}. Run cell 1 first.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


def _run(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout[-4000:])
    if result.returncode != 0 and result.stderr.strip():
        print(result.stderr[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")
    return result


def _ensure_system_packages() -> None:
    has_ffmpeg = bool(shutil.which("ffmpeg"))
    has_espeak = bool(shutil.which("espeak-ng") or shutil.which("espeak"))
    if has_ffmpeg and has_espeak:
        print("System packages: ffmpeg/espeak already available")
        return

    packages: list[str] = []
    if not has_ffmpeg:
        packages.append("ffmpeg")
    if not shutil.which("espeak-ng"):
        packages.append("espeak-ng")

    last_error: Exception | None = None
    for prefix in ([], ["sudo"]):
        try:
            _run(prefix + ["apt-get", "update", "-qq"])
            if packages:
                _run(prefix + ["apt-get", "install", "-y", "-qq", *packages])
            print("Installed:", ", ".join(packages) or "(none)")
            return
        except Exception as exc:  # noqa: BLE001
            last_error = exc

    if not shutil.which("ffmpeg"):
        raise RuntimeError("ffmpeg is missing and could not be installed.") from last_error
    if not (shutil.which("espeak-ng") or shutil.which("espeak")):
        raise RuntimeError("espeak-ng is missing and could not be installed.") from last_error


_ensure_system_packages()

_run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "TTS",
        "audiocraft",
        "langchain",
        "langchain-community",
        "langchain-text-splitters",
    ],
    check=False,
)
_run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])
_run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])

# Drop stale imports so this kernel picks up newly installed packages.
for name in list(sys.modules):
    if (
        name == "TTS"
        or name.startswith("TTS.")
        or name.startswith("transformers")
        or name.startswith("tokenizers")
        or name.startswith("langchain")
        or name.startswith("langgraph")
        or name.startswith("podcast_gen_agent")
    ):
        del sys.modules[name]


def _verify() -> None:
    from packaging import version

    print("Verifying installation in-kernel...")

    import torch

    print("  torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError(
            "GPU is not available. Enable a GPU for this notebook, "
            "restart the session, then re-run cells 1 and 2."
        )
    print("  GPU:", torch.cuda.get_device_name(0))

    if not shutil.which("ffmpeg"):
        raise RuntimeError("ffmpeg is missing from PATH")
    if not (shutil.which("espeak-ng") or shutil.which("espeak")):
        raise RuntimeError("espeak-ng/espeak is missing from PATH")
    print("  ffmpeg/espeak: OK")

    import tokenizers
    import transformers

    if version.parse(transformers.__version__) < version.parse("4.57.5"):
        raise RuntimeError(f"transformers too old: {transformers.__version__}")
    if version.parse(tokenizers.__version__) < version.parse("0.22.0"):
        raise RuntimeError(f"tokenizers too old: {tokenizers.__version__}")
    print("  transformers:", transformers.__version__)
    print("  tokenizers:", tokenizers.__version__)

    from podcast_gen_agent.compat import ensure_langchain_compat, ensure_transformers_compat

    ensure_transformers_compat()
    ensure_langchain_compat()

    from langchain_core.globals import get_debug

    assert get_debug() in (True, False)
    print("  langchain-core: OK")

    from TTS.api import TTS  # noqa: F401

    print("  XTTS import: OK")

    from transformers import MusicgenForConditionalGeneration  # noqa: F401

    print("  MusicGen import: OK")

    from podcast_gen_agent.graph import get_graph

    get_graph()
    print("  LangGraph compile: OK")


_verify()

import podcast_gen_agent

print("All dependencies and pipeline imports are ready.")
print("Kernel package path:", podcast_gen_agent.__file__)


In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Enable a GPU for this notebook, then run again from cell 1."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Change only TOPIC and DURATION for each podcast.
import json
import os
import sys
from pathlib import Path

def _work_root() -> Path:
    for candidate in (Path("/kaggle/working"), Path("/content")):
        if candidate.is_dir():
            return candidate
    return Path.cwd()


def _resolve_repo_dir() -> Path:
    env_repo = os.environ.get("PODCAST_GEN_REPO")
    if env_repo:
        candidate = Path(env_repo).resolve()
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate
    for candidate in (
        _work_root() / "PodcastGen-Agent",
        Path.cwd(),
        Path.cwd().parent,
    ):
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate.resolve()
    raise RuntimeError("Repository not found. Run cell 1 first.")


def _prepare_repo(repo_dir: Path | None = None) -> Path:
    root = _resolve_repo_dir() if repo_dir is None else Path(repo_dir).resolve()
    os.chdir(root)
    os.environ["PODCAST_GEN_REPO"] = str(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root


def _find_latest_podcast_mp3() -> Path | None:
    roots: list[Path] = []
    seen: set[str] = set()

    def add(path: Path | str) -> None:
        resolved = Path(path).resolve()
        key = str(resolved)
        if key in seen:
            return
        seen.add(key)
        roots.append(resolved)

    repo = Path(os.environ.get("PODCAST_GEN_REPO", Path.cwd())).resolve()
    add(repo / "outputs")
    add(Path.cwd() / "outputs")
    add(_work_root() / "PodcastGen-Agent" / "outputs")
    for content_root in (Path("/content"), Path("/kaggle/working")):
        if not content_root.exists():
            continue
        for manifest in content_root.rglob("run_manifest.json"):
            add(manifest.parent)

    best: Path | None = None
    best_mtime = -1.0
    for root in roots:
        if not root.exists():
            continue
        mp3_files = [p for p in root.rglob("podcast_*.mp3") if p.stat().st_size > 0]
        if not mp3_files:
            mp3_files = [p for p in root.rglob("*.mp3") if p.stat().st_size > 0]
        for path in mp3_files:
            mtime = path.stat().st_mtime
            if mtime > best_mtime:
                best_mtime = mtime
                best = path
    return best


def _download_or_link_file(path: Path) -> None:
    import importlib

    resolved = path.resolve()

    try:
        files_mod = importlib.import_module("google.colab.files")
        files_mod.download(str(resolved))
        return
    except Exception:
        pass

    work = _work_root()
    dest = work / resolved.name
    if resolved != dest.resolve():
        dest.write_bytes(resolved.read_bytes())
        print(f"Copied to {dest}")

    try:
        from IPython.display import FileLink, display

        try:
            link_target = str(dest.relative_to(Path.cwd()))
        except ValueError:
            link_target = dest.name
        display(FileLink(link_target))
        print("Use the link above, or download the file from the notebook output files.")
    except Exception:
        print(f"File ready at: {dest}")

REPO_DIR = _prepare_repo()

TOPIC = "LoRA and QLoRA"
DURATION = 2
RUN_ID = ""

assert TOPIC.strip(), "TOPIC must not be empty"
assert 1 <= DURATION <= 30, "DURATION must be between 1 and 30 minutes"

os.environ["COQUI_TOS_AGREED"] = "1"
os.environ["PODCAST_LOG_LEVEL"] = "INFO"
os.environ["PODCAST_MIN_GPU_FREE_MB"] = "128"
os.environ["PODCAST_MAX_SCRIPT_LINES"] = "12"

# Optional voice presets / cloning / research API keys:
# os.environ["PODCAST_HOST_VOICE"] = "Craig Gutsy"
# os.environ["PODCAST_GUEST_VOICE"] = "Gracie Wise"
# os.environ["PODCAST_HOST_SPEAKER_WAV"] = str(REPO_DIR / "voices" / "host_ref.wav")
# os.environ["PODCAST_GUEST_SPEAKER_WAV"] = str(REPO_DIR / "voices" / "guest_ref.wav")
# os.environ["PODCAST_RESEARCH_PROVIDERS"] = "wikipedia,arxiv,tavily,brave,duckduckgo"
# os.environ["PODCAST_TAVILY_API_KEY"] = "tvly-xxxxxxxx"
# os.environ["PODCAST_BRAVE_API_KEY"] = "BSA-xxxxxxxx"

argv = ["podcast_gen_agent.main", TOPIC, "--duration", str(DURATION)]
if RUN_ID.strip():
    argv.extend(["--resume", RUN_ID.strip()])
sys.argv = argv

print("Repo:", REPO_DIR)
print("Running:", " ".join(argv[1:]))

try:
    from podcast_gen_agent.compat import ensure_langchain_compat
    from podcast_gen_agent.main import main

    ensure_langchain_compat()
    main()
except SystemExit as exc:
    if exc.code not in (0, None):
        from podcast_gen_agent.config import settings

        outputs = settings.output_dir.resolve()
        manifests = sorted(
            outputs.glob("*/run_manifest.json"),
            key=lambda p: p.stat().st_mtime,
        )
        if manifests:
            manifest = json.loads(manifests[-1].read_text(encoding="utf-8"))
            print("Latest manifest:", manifests[-1])
            print("Run id:", manifest.get("run_id"))
            print("Status:", manifest.get("status"))
            print("Error:", manifest.get("error"))
        raise

latest = _find_latest_podcast_mp3()
assert latest is not None and latest.stat().st_size > 0, (
    "Generation finished without producing a non-empty MP3. Check the logs above."
)
print(f"SUCCESS: {latest} ({latest.stat().st_size / 1_000_000:.1f} MB)")


In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import Audio, display

def _work_root() -> Path:
    for candidate in (Path("/kaggle/working"), Path("/content")):
        if candidate.is_dir():
            return candidate
    return Path.cwd()


def _resolve_repo_dir() -> Path:
    env_repo = os.environ.get("PODCAST_GEN_REPO")
    if env_repo:
        candidate = Path(env_repo).resolve()
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate
    for candidate in (
        _work_root() / "PodcastGen-Agent",
        Path.cwd(),
        Path.cwd().parent,
    ):
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate.resolve()
    raise RuntimeError("Repository not found. Run cell 1 first.")


def _prepare_repo(repo_dir: Path | None = None) -> Path:
    root = _resolve_repo_dir() if repo_dir is None else Path(repo_dir).resolve()
    os.chdir(root)
    os.environ["PODCAST_GEN_REPO"] = str(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root


def _find_latest_podcast_mp3() -> Path | None:
    roots: list[Path] = []
    seen: set[str] = set()

    def add(path: Path | str) -> None:
        resolved = Path(path).resolve()
        key = str(resolved)
        if key in seen:
            return
        seen.add(key)
        roots.append(resolved)

    repo = Path(os.environ.get("PODCAST_GEN_REPO", Path.cwd())).resolve()
    add(repo / "outputs")
    add(Path.cwd() / "outputs")
    add(_work_root() / "PodcastGen-Agent" / "outputs")
    for content_root in (Path("/content"), Path("/kaggle/working")):
        if not content_root.exists():
            continue
        for manifest in content_root.rglob("run_manifest.json"):
            add(manifest.parent)

    best: Path | None = None
    best_mtime = -1.0
    for root in roots:
        if not root.exists():
            continue
        mp3_files = [p for p in root.rglob("podcast_*.mp3") if p.stat().st_size > 0]
        if not mp3_files:
            mp3_files = [p for p in root.rglob("*.mp3") if p.stat().st_size > 0]
        for path in mp3_files:
            mtime = path.stat().st_mtime
            if mtime > best_mtime:
                best_mtime = mtime
                best = path
    return best


def _download_or_link_file(path: Path) -> None:
    import importlib

    resolved = path.resolve()

    try:
        files_mod = importlib.import_module("google.colab.files")
        files_mod.download(str(resolved))
        return
    except Exception:
        pass

    work = _work_root()
    dest = work / resolved.name
    if resolved != dest.resolve():
        dest.write_bytes(resolved.read_bytes())
        print(f"Copied to {dest}")

    try:
        from IPython.display import FileLink, display

        try:
            link_target = str(dest.relative_to(Path.cwd()))
        except ValueError:
            link_target = dest.name
        display(FileLink(link_target))
        print("Use the link above, or download the file from the notebook output files.")
    except Exception:
        print(f"File ready at: {dest}")

REPO_DIR = _prepare_repo()
latest = _find_latest_podcast_mp3()
if latest:
    size_mb = latest.stat().st_size / 1_000_000
    print(f"Playing: {latest} ({size_mb:.1f} MB)")
    display(Audio(data=latest.read_bytes()))
else:
    print("No podcast mp3 found.")
    print("cwd:", os.getcwd())
    print("Repo:", REPO_DIR)
    print("Re-run cells 1, 2, and 4 if generation has not completed yet.")


In [ ]:
import os
import sys
from pathlib import Path

def _work_root() -> Path:
    for candidate in (Path("/kaggle/working"), Path("/content")):
        if candidate.is_dir():
            return candidate
    return Path.cwd()


def _resolve_repo_dir() -> Path:
    env_repo = os.environ.get("PODCAST_GEN_REPO")
    if env_repo:
        candidate = Path(env_repo).resolve()
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate
    for candidate in (
        _work_root() / "PodcastGen-Agent",
        Path.cwd(),
        Path.cwd().parent,
    ):
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate.resolve()
    raise RuntimeError("Repository not found. Run cell 1 first.")


def _prepare_repo(repo_dir: Path | None = None) -> Path:
    root = _resolve_repo_dir() if repo_dir is None else Path(repo_dir).resolve()
    os.chdir(root)
    os.environ["PODCAST_GEN_REPO"] = str(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root


def _find_latest_podcast_mp3() -> Path | None:
    roots: list[Path] = []
    seen: set[str] = set()

    def add(path: Path | str) -> None:
        resolved = Path(path).resolve()
        key = str(resolved)
        if key in seen:
            return
        seen.add(key)
        roots.append(resolved)

    repo = Path(os.environ.get("PODCAST_GEN_REPO", Path.cwd())).resolve()
    add(repo / "outputs")
    add(Path.cwd() / "outputs")
    add(_work_root() / "PodcastGen-Agent" / "outputs")
    for content_root in (Path("/content"), Path("/kaggle/working")):
        if not content_root.exists():
            continue
        for manifest in content_root.rglob("run_manifest.json"):
            add(manifest.parent)

    best: Path | None = None
    best_mtime = -1.0
    for root in roots:
        if not root.exists():
            continue
        mp3_files = [p for p in root.rglob("podcast_*.mp3") if p.stat().st_size > 0]
        if not mp3_files:
            mp3_files = [p for p in root.rglob("*.mp3") if p.stat().st_size > 0]
        for path in mp3_files:
            mtime = path.stat().st_mtime
            if mtime > best_mtime:
                best_mtime = mtime
                best = path
    return best


def _download_or_link_file(path: Path) -> None:
    import importlib

    resolved = path.resolve()

    try:
        files_mod = importlib.import_module("google.colab.files")
        files_mod.download(str(resolved))
        return
    except Exception:
        pass

    work = _work_root()
    dest = work / resolved.name
    if resolved != dest.resolve():
        dest.write_bytes(resolved.read_bytes())
        print(f"Copied to {dest}")

    try:
        from IPython.display import FileLink, display

        try:
            link_target = str(dest.relative_to(Path.cwd()))
        except ValueError:
            link_target = dest.name
        display(FileLink(link_target))
        print("Use the link above, or download the file from the notebook output files.")
    except Exception:
        print(f"File ready at: {dest}")

REPO_DIR = _prepare_repo()
latest = _find_latest_podcast_mp3()
if latest:
    print(f"Downloading: {latest}")
    _download_or_link_file(latest)
else:
    print("No podcast file found.")
    print("cwd:", os.getcwd())
    print("Repo:", REPO_DIR)
    print("Re-run cells 1, 2, and 4, or download the MP3 from the output files.")
